# v2 Feature Analysis: VIF & SHAP

Run VIF multicollinearity analysis and SHAP feature importance on SP_ENHANCED feature set
to identify features to prune before final model training in Plan 06-02.

In [ ]:
# Cell 1 - Build v2 Feature Store (skip if already exists)
from pathlib import Path
import pandas as pd
from src.features.feature_builder import FeatureBuilder
from src.models.feature_sets import SP_ENHANCED_FEATURE_COLS

FEATURE_STORE_PATH = "data/features/feature_store_v2.parquet"

if Path(FEATURE_STORE_PATH).exists():
    df = pd.read_parquet(FEATURE_STORE_PATH)
    print(f"Feature store v2 already exists -- skipping build.")
else:
    print("Building v2 feature store (10-30 min depending on cache)...")
    builder = FeatureBuilder(seasons=list(range(2015, 2026)))
    df = builder.build_and_save_v2(FEATURE_STORE_PATH)

print(f"Feature store v2: {df.shape[0]} games, {df.shape[1]} columns")
print(f"Seasons: {sorted(df['season'].unique())}")
print(f"SP_ENHANCED NaN counts:\n{df[SP_ENHANCED_FEATURE_COLS].isna().sum()}")

In [ ]:
# Cell 2 - VIF Analysis
from src.models.vif import compute_vif
from src.models.feature_sets import SP_ENHANCED_FEATURE_COLS

# Use NaN-dropped training data for VIF (same filter as backtest)
df_clean = df[df['rolling_ops_diff'].notna()].copy()
X_vif = df_clean[SP_ENHANCED_FEATURE_COLS].dropna()
vif_results = compute_vif(X_vif)
vif_results.to_csv("data/results/vif_analysis.csv", index=False)
print("VIF Analysis Results:")
print(vif_results.to_string(index=False))
high_vif = vif_results[vif_results['VIF'] > 10]
print(f"\nFeatures with VIF > 10: {len(high_vif)}")
if len(high_vif) > 0:
    print(high_vif.to_string(index=False))

In [ ]:
# Cell 3 - Iterative VIF Pruning
pruned_cols = list(SP_ENHANCED_FEATURE_COLS)
dropped_features = []
while True:
    X_check = df_clean[pruned_cols].dropna()
    vif_check = compute_vif(X_check)
    worst = vif_check.iloc[0]  # sorted descending
    if worst['VIF'] <= 10:
        break
    print(f"Dropping {worst['feature']} (VIF={worst['VIF']})")
    dropped_features.append(worst['feature'])
    pruned_cols.remove(worst['feature'])
print(f"Dropped features: {dropped_features}")
print(f"Remaining: {len(pruned_cols)} features")
print(f"\nFinal VIF (all <= 10):")
final_vif = compute_vif(df_clean[pruned_cols].dropna())
print(final_vif.to_string(index=False))

In [ ]:
# Cell 4 - Preliminary XGBoost Training for SHAP
from src.models.train import make_xgb_model
from src.models.shap_analysis import compute_shap_importance
from src.models.feature_sets import TARGET_COL

# Train preliminary XGBoost on all training data (2015-2023) for SHAP
train_mask = df_clean['season'].isin(list(range(2015, 2024)))
X_train = df_clean.loc[train_mask, pruned_cols].dropna()
y_train = df_clean.loc[train_mask & df_clean[pruned_cols].notna().all(axis=1), TARGET_COL]
# Align indices
common_idx = X_train.index.intersection(y_train.index)
X_train = X_train.loc[common_idx]
y_train = y_train.loc[common_idx]

# Early stopping split
n_val = int(len(X_train) * 0.2)
xgb_model = make_xgb_model()
xgb_model.fit(
    X_train.iloc[:-n_val], y_train.iloc[:-n_val],
    eval_set=[(X_train.iloc[-n_val:], y_train.iloc[-n_val:])],
    verbose=False
)

shap_results = compute_shap_importance(xgb_model, X_train, pruned_cols)
print("SHAP Feature Importance:")
print(shap_results.to_string(index=False))

near_zero = shap_results[shap_results['pct_of_total'] < 0.001]
print(f"\nNear-zero features (< 0.1% of SHAP budget): {len(near_zero)}")
if len(near_zero) > 0:
    print(near_zero.to_string(index=False))

In [ ]:
# Cell 5 - Final Feature Set Documentation
final_sp_enhanced = [c for c in pruned_cols if c not in near_zero['feature'].tolist()]
print(f"Final SP_ENHANCED features ({len(final_sp_enhanced)}):")
for c in final_sp_enhanced:
    print(f"  - {c}")
print(f"\nDropped by VIF (>10): {dropped_features}")
print(f"Dropped by SHAP (<0.1%): {near_zero['feature'].tolist() if len(near_zero) > 0 else 'none'}")
print(f"\nSummary: 20 original SP_ENHANCED features -> {len(final_sp_enhanced)} after pruning")
print(f"  - VIF removed {len(dropped_features)} collinear features")
print(f"  - SHAP removed {len(near_zero)} near-zero importance features")